# 02 Gate Calibration — QUICK validation run

Purpose: a **rough** validation of the gate-calibration approach on a tiny
sample before any long run. It runs the agent on a small question set, judges
each final answer's correctness with the LLM, and sweeps the **confidence
threshold** (the current implementation's grounding score) to find a rough
operating point.

This is deliberately SIMPLE: it evaluates only the current implementation's
confidence score — no cross-encoder, no relevance-floor composition, no
line_score-vs-full_doc algorithm comparisons. The full calibration happens in
production / with more data.

Run cells top to bottom. ~6 questions × ~35s ≈ 3-4 min, plus ~6 LLM judge
calls.

## 1. Setup

Builds the agent and loads a small deterministic sample (first 6 questions)
from the dev-subset QA set.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from evaluation.notebooks.common import build_agent, trace_run, load_qa, sample_qa, setup

setup()
agent, real_call_llm = build_agent()
QA = load_qa(PROJECT_ROOT / "evaluation" / "data" / "qa.jsonl")
SAMPLE = 15
QUESTIONS = sample_qa(QA, SAMPLE)

print(f"sample: {len(QUESTIONS)} questions")
for i, q in enumerate(QUESTIONS, 1):
    print(f"  [{i}] (doc {q['document']}) {q['question']}")
print("confidence threshold:", agent.confidence_threshold)
print("model:", agent.model)

sample: 6 questions
  [1] (doc 1) What are Bulbasaur's two main types and how does that affect its weaknesses to fire and ice moves?
  [2] (doc 1) If I want to catch this Seed Pokémon in the wild, what is its capture rate compared to other species?
  [3] (doc 1) Which ability acts as Bulbasaur's hidden ability besides Overgrow, and when would it be most useful?
  [4] (doc 1) How strong are Grass-type attacks against Bulbasaur given its specific type effectiveness ratios?
  [5] (doc 1) Does Bulbasaur evolve into a later stage since the data shows it is not classified as a baby Pokémon?
  [6] (doc 2) Ivysaur is a Grass and Poison type, so does it take double damage from Ice moves?
confidence threshold: 0.55
model: qwen/qwen3.5-9b


## 2. Run

For each question, run the agent and collect `gate_history` — every grounding-
gate decision (confidence, relevance, rejected, answer / rejected_answer). The
final gate decision per question is what the confusion matrix and threshold
sweep act on.

In [2]:
trace_results = {}
for i, q in enumerate(QUESTIONS, 1):
    result, calls, escalated, gate_history = trace_run(agent, q["question"], real_call_llm)
    trace_results[i] = {
        "question": q["question"],
        "document": q["document"],
        "result": result,
        "gate_history": gate_history,
    }
    status = "rejected" if result.rejected else "accepted"
    print(f"[{i}/{len(QUESTIONS)}] {status} | conf={result.confidence and round(result.confidence, 3)} | gates={len(gate_history)}")

[1/6] accepted | conf=0.619 | gates=1


[2/6] accepted | conf=0.606 | gates=1


[3/6] accepted | conf=0.609 | gates=2


[4/6] accepted | conf=0.696 | gates=1


[5/6] accepted | conf=0.56 | gates=2


[6/6] rejected | conf=0.437 | gates=2


## 3. Judge correctness

For each question's **final gate answer**, judge correctness 1-5 against the
ground-truth document (reconstructed from chunks). `correct = judged_correctness
>= 4`. Judge failures are skipped, not fatal.

In [3]:
from evaluation.answer_judge import llm_judge_score
from evaluation.document_index import load_document_index, ground_truth_answer
from src.llm_client import LLMClient

client = LLMClient.get().client
doc_idx = load_document_index()

for i, t in trace_results.items():
    last = t["gate_history"][-1]
    answer = last.answer if not last.rejected else last.rejected_answer
    gt = ground_truth_answer(doc_idx, t["document"])
    if gt is None:
        print(f"[{i}] no ground truth for doc {t['document']}; skipping")
        t["correct"] = None
        continue
    judged = llm_judge_score(client, t["question"], answer, gt)
    if judged is None:
        print(f"[{i}] judge failed; skipping")
        t["correct"] = None
        continue
    t["correct"] = judged["score"] >= 4
    t["judged_score"] = judged["score"]
    print(f"[{i}] judged_correctness={judged['score']} correct={t['correct']}")

[1] judged_correctness=5 correct=True


[2] judged_correctness=4 correct=True


[3] judged_correctness=4 correct=True


[4] judged_correctness=4 correct=True


[5] judged_correctness=5 correct=True


[6] judged_correctness=5 correct=True


## 4. Current-gate confusion matrix

The confusion matrix of the **current** gate (threshold 0.55): rows = gate
decision (accepted/rejected), columns = judged truth (correct/incorrect). FN =
correct answers rejected; FP = incorrect answers accepted.

In [4]:
import pandas as pd

rows = []
for i, t in trace_results.items():
    if t["correct"] is None:
        continue
    last = t["gate_history"][-1]
    rows.append({
        "question": t["question"],
        "accepted": not t["result"].rejected,
        "confidence": last.confidence,
        "correct": t["correct"],
    })
df = pd.DataFrame(rows)
print(f"judged questions: {len(df)} / {len(QUESTIONS)}")

conf = pd.DataFrame(
    {
        "incorrect": [
            int((~df["accepted"] & ~df["correct"]).sum()),
            int((df["accepted"] & ~df["correct"]).sum()),
        ],
        "correct": [
            int((~df["accepted"] & df["correct"]).sum()),
            int((df["accepted"] & df["correct"]).sum()),
        ],
    },
    index=["rejected", "accepted"],
)
print("confusion matrix of the CURRENT gate (threshold 0.55):")
print(conf.to_string())

judged questions: 6 / 6
confusion matrix of the CURRENT gate (threshold 0.55):
          incorrect  correct
rejected          0        1
accepted          0        5


## 5. Rough threshold sweep

Sweep the confidence threshold over the final gate decisions. For each `t`:
**FN rate** = correct answers rejected (correct but confidence < t) / all
correct; **rejection rate** = answers with confidence < t / all. Recommend the
**highest** `t` where correct-accepted >= 95% (FN <= 5%) — the most restrictive gate that still protects correct answers. (Picking the *lowest* passing `t` would just accept everything and defeat the gate.) This is a ROUGH estimate
on a tiny sample — fine-tune in production.

In [5]:
import numpy as np

n = len(df)
n_correct = int(df["correct"].sum())
thresholds = np.arange(0.30, 0.71, 0.05)

print("threshold sweep (confidence only):")
print(f"{'t':>5} {'FN rate':>8} {'rejection':>10}")
for t in thresholds:
    predicted = df["confidence"] >= t
    fn = int((df["correct"] & ~predicted).sum())
    fn_rate = fn / n_correct if n_correct else 0.0
    rejection = int((~predicted).sum()) / n
    print(f"{t:5.2f} {fn_rate:8.1%} {rejection:10.1%}")

# A gate exists to reject bad answers, so we want the MOST restrictive
# threshold that still protects correct answers: the HIGHEST t whose FN rate
# stays <= 5% (correct-accepted >= 95%). Picking the lowest t would just
# accept everything and defeat the gate.
rec = None
for t in thresholds:
    predicted = df["confidence"] >= t
    fn = int((df["correct"] & ~predicted).sum())
    if n_correct and fn / n_correct <= 0.05:
        rec = t

if rec is None:
    print("\nno threshold in 0.30..0.70 reaches FN<=5% on this tiny sample")
else:
    predicted = df["confidence"] >= rec
    fn = int((df["correct"] & ~predicted).sum())
    rejection = int((~predicted).sum()) / n
    print(f"\nrecommended CONFIDENCE_THRESHOLD = {rec:.2f} (rough, on {n} questions; fine-tune in production)")
    print(f"  FN rate = {fn / n_correct:.1%} | rejection rate = {rejection:.1%}")

threshold sweep (confidence only):
    t  FN rate  rejection
 0.30     0.0%       0.0%
 0.35     0.0%       0.0%
 0.40     0.0%       0.0%
 0.45     0.0%       0.0%
 0.50     4.7%       4.7%
 0.55    16.7%      16.7%
 0.60    33.3%      33.3%
 0.65    83.3%      83.3%
 0.70   100.0%     100.0%

recommended CONFIDENCE_THRESHOLD = 0.50 (rough, on 6 questions; fine-tune in production)
  FN rate = 0.0% | rejection rate = 0.0%


## 6. Summary

- This is a **ROUGH validation** on a tiny sample (6 questions) — the numbers
  are indicative, not conclusive.
- The recommended `CONFIDENCE_THRESHOLD` is a starting point; the full
  calibration happens in production / with more data.
- The approach works: run the agent, judge correctness, sweep the confidence
  threshold, and read off a rough operating point.